<a href="https://colab.research.google.com/github/vifirsanova/ML-2026-pt-2/blob/main/rnn_tutorial.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# PyTorch RNN: полный туториал по классификации текста

В этом ноутбуке мы пройдём полный цикл работы с рекуррентной нейросетью: от загрузки данных и предобработки до обучения, визуализации и настройки гиперпараметров. В конце каждого блока есть задания на 5–10 минут, где вы сможете самостоятельно изменить параметры и проанализировать результаты.

**План:**
1. Загрузка данных через Hugging Face `datasets`
2. Предобработка текста (токенизация, словарь, паддинг)
3. Построение RNN-классификатора
4. Обучение и валидация
5. Оценка: кривые обучения, матрица ошибок, classification report
6. Три практических задания с плейсхолдерами для гиперпараметров

## 1. Окружение

Устанавливаем и импортируем всё необходимое. Датасет будем брать через `datasets` от Hugging Face — это позволяет скачать данные автоматически, без ручной подготовки.

In [ ]:
!pip install -q datasets torchtext scikit-learn seaborn tqdm matplotlib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.2/64.2 kB 1.7 MB/s eta 0:00:00


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchtext.data.utils import get_tokenizer
from torchtext.vocab import build_vocab_from_iterator
from datasets import load_dataset
from collections import Counter
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report
import numpy as np
from tqdm import tqdm

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## 2. Загрузка данных

Используем датасет **AG News** (классификация новостных тем, 4 класса). Он доступен на Hugging Face Hub и скачивается автоматически.

In [ ]:
# Загружаем датасет AG News через Hugging Face
dataset = load_dataset("ag_news")

print(dataset)
print("\nПример:", dataset['train'][0])

In [ ]:
train_data = list(zip(dataset['train']['label'], dataset['train']['text']))
test_data = list(zip(dataset['test']['label'], dataset['test']['text']))

print(f"Обучающих примеров: {len(train_data)}")
print(f"Тестовых примеров: {len(test_data)}")
print(f"\nРаспределение меток (обучение): {Counter([label for label, _ in train_data])}")

### Визуализация данных

Посмотрим на баланс классов и распределение длин текстов. Это поможет выбрать разумное значение `MAX_LEN` для паддинга.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

labels = [label for label, _ in train_data]
axes[0].bar(Counter(labels).keys(), Counter(labels).values(), color='steelblue')
axes[0].set_xlabel('Class Label')
axes[0].set_ylabel('Count')
axes[0].set_title('Class Distribution (Train)')
axes[0].set_xticks([0, 1, 2, 3])

lengths = [len(text.split()) for _, text in train_data[:500]]
axes[1].hist(lengths, bins=30, color='coral', edgecolor='white')
axes[1].set_xlabel('Text Length (words)')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Text Length Distribution')

plt.tight_layout()
plt.show()

print("\n=== Примеры ===")
for i in range(3):
    label, text = train_data[i]
    print(f"[Label {label}] {text[:150]}...")

## 3. Предобработка текста

Токенизируем тексты, строим словарь и приводим последовательности к одной длине. Здесь есть плейсхолдеры для изменения параметров.

In [ ]:
tokenizer = get_tokenizer('basic_english')

def yield_tokens(data_iter):
    for label, text in data_iter:
        yield tokenizer(text)

# ============================================
# Задание 1: параметры построения словаря
# Попробуйте min_freq = 2 или 10. Как меняются размер словаря и качество модели?
# Почему редкие слова могут как помогать, так и мешать?
# Документация: https://pytorch.org/text/stable/vocab.html#build-vocab-from-iterator
# ============================================
MIN_FREQ = 5  # <-- меняйте здесь

vocab = build_vocab_from_iterator(
    yield_tokens(train_data),
    specials=['<unk>', '<pad>'],
    min_freq=MIN_FREQ
)
vocab.set_default_index(vocab['<unk>'])

PAD_IDX = vocab['<pad>']
print(f"Размер словаря: {len(vocab)}")

In [ ]:
# ============================================
# Задание 2: длина последовательности (обрезка)
# Попробуйте MAX_LEN = 128, 256, 512. Как меняются скорость обучения и точность?
# Почему у RNN может не быть выигрыша от очень длинных последовательностей?
# ============================================
MAX_LEN = 256  # <-- меняйте здесь

def text_pipeline(text):
    tokens = tokenizer(text)
    ids = [vocab[token] for token in tokens]
    if len(ids) > MAX_LEN:
        ids = ids[:MAX_LEN]
    return ids

def label_pipeline(label):
    return label

class AGNewsDataset(torch.utils.data.Dataset):
    def __init__(self, data, text_pipeline, label_pipeline, max_len):
        self.data = data
        self.text_pipeline = text_pipeline
        self.label_pipeline = label_pipeline
        self.max_len = max_len

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        label, text = self.data[idx]
        ids = self.text_pipeline(text)
        return torch.tensor(ids, dtype=torch.long), torch.tensor(self.label_pipeline(label), dtype=torch.long)

def collate_batch(batch):
    text_list, label_list = [], []
    for _text, _label in batch:
        text_list.append(_text)
        label_list.append(_label)
    padded = nn.utils.rnn.pad_sequence(text_list, batch_first=True, padding_value=PAD_IDX)
    return padded, torch.stack(label_list)

train_dataset = AGNewsDataset(train_data, text_pipeline, label_pipeline, MAX_LEN)
test_dataset = AGNewsDataset(test_data, text_pipeline, label_pipeline, MAX_LEN)

In [ ]:
# ============================================
# Задание 3: размер батча
# Попробуйте BATCH_SIZE = 32, 64, 128. Как это влияет на стабильность обучения и скорость?
# Документация: https://pytorch.org/docs/stable/data.html#torch.utils.data.DataLoader
# ============================================
BATCH_SIZE = 64  # <-- меняйте здесь

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          collate_fn=collate_batch, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False,
                         collate_fn=collate_batch, num_workers=2)

print(f"Количество батчей в обучении: {len(train_loader)}")

## 4. Модель

Стандартная схема: `Embedding → RNN → Linear`. На выходе берём скрытое состояние последнего временного шага и подаём его в полносвязный слой.

In [ ]:
class RNNClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_classes,
                 num_layers=1, dropout=0.2, bidirectional=False):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=PAD_IDX)
        self.rnn = nn.RNN(
            input_size=embed_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0,
            bidirectional=bidirectional
        )
        rnn_output_dim = hidden_dim * 2 if bidirectional else hidden_dim
        self.fc = nn.Linear(rnn_output_dim, num_classes)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        embedded = self.dropout(self.embedding(x))
        output, hidden = self.rnn(embedded)
        if self.rnn.bidirectional:
            last_hidden = torch.cat([hidden[-2], hidden[-1]], dim=1)
        else:
            last_hidden = hidden[-1]
        return self.fc(self.dropout(last_hidden))

In [ ]:
# ============================================
# Задание 4: гиперпараметры модели
# Попробуйте разные комбинации и посмотрите на точность на валидации:
#   EMBED_DIM: 64, 128, 256
#   HIDDEN_DIM: 64, 128, 256
#   NUM_LAYERS: 1, 2
#   BIDIRECTIONAL: True / False
# Как размер скрытого состояния связан с числом параметров и качеством?
# ============================================
EMBED_DIM = 128       # <-- меняйте здесь
HIDDEN_DIM = 128      # <-- меняйте здесь
NUM_LAYERS = 1        # <-- меняйте здесь
BIDIRECTIONAL = False # <-- меняйте здесь (True/False)
DROPOUT = 0.2

model = RNNClassifier(
    vocab_size=len(vocab),
    embed_dim=EMBED_DIM,
    hidden_dim=HIDDEN_DIM,
    num_classes=4,
    num_layers=NUM_LAYERS,
    dropout=DROPOUT,
    bidirectional=BIDIRECTIONAL
).to(device)

print(model)
print(f"Количество обучаемых параметров: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

## 5. Обучение и валидация

In [ ]:
def train_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss, total_correct, total_samples = 0, 0, 0
    for texts, labels in tqdm(loader, desc="Training"):
        texts, labels = texts.to(device), labels.to(device)
        optimizer.zero_grad()
        logits = model(texts)
        loss = criterion(logits, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        total_loss += loss.item() * labels.size(0)
        preds = logits.argmax(dim=1)
        total_correct += (preds == labels).sum().item()
        total_samples += labels.size(0)
    return total_loss / total_samples, total_correct / total_samples

def evaluate(model, loader, criterion):
    model.eval()
    total_loss, total_correct, total_samples = 0, 0, 0
    all_preds, all_labels = [], []
    with torch.no_grad():
        for texts, labels in tqdm(loader, desc="Evaluating"):
            texts, labels = texts.to(device), labels.to(device)
            logits = model(texts)
            loss = criterion(logits, labels)
            total_loss += loss.item() * labels.size(0)
            preds = logits.argmax(dim=1)
            total_correct += (preds == labels).sum().item()
            total_samples += labels.size(0)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    return total_loss / total_samples, total_correct / total_samples, all_preds, all_labels

In [ ]:
# ============================================
# Задание 5: параметры оптимизатора
# Попробуйте LEARNING_RATE = 1e-3, 5e-4, 1e-4.
# Как скорость обучения влияет на сходимость и стабильность?
# Документация: https://pytorch.org/docs/stable/optim.html#torch.optim.Adam
# ============================================
LEARNING_RATE = 1e-3  # <-- меняйте здесь
NUM_EPOCHS = 5        # <-- меняйте здесь

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}

for epoch in range(NUM_EPOCHS):
    train_loss, train_acc = train_epoch(model, train_loader, optimizer, criterion)
    val_loss, val_acc, _, _ = evaluate(model, test_loader, criterion)
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    print(f"Epoch {epoch+1}/{NUM_EPOCHS} | "
          f"Train Loss: {train_loss:.4f} Acc: {train_acc:.4f} | "
          f"Val Loss: {val_loss:.4f} Acc: {val_acc:.4f}")

## 6. Визуализация и анализ

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history['train_loss'], label='Train Loss', marker='o')
axes[0].plot(history['val_loss'], label='Val Loss', marker='s')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Loss Curve')
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].plot(history['train_acc'], label='Train Acc', marker='o')
axes[1].plot(history['val_acc'], label='Val Acc', marker='s')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].set_title('Accuracy Curve')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
val_loss, val_acc, all_preds, all_labels = evaluate(model, test_loader, criterion)

cm = confusion_matrix(all_labels, all_preds)
class_names = ['World', 'Sports', 'Business', 'Sci/Tech']

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names)
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Confusion Matrix (Test Set)')
plt.tight_layout()
plt.show()

print("\n=== Classification Report ===")
print(classification_report(all_labels, all_preds, target_names=class_names))

## 7. Практические задания

### Задание 1: Влияние размера словаря (5 минут)

Вернитесь в блок с построением словаря (`MIN_FREQ = 5`) и измените `MIN_FREQ` на `2`. Перезапустите все ячейки ниже и сравните:

- Насколько изменился размер словаря?
- Изменилась ли точность на валидации?
- Почему редкие слова могут как помогать, так и мешать модели? Подумайте о количестве примеров, в которых встречается каждое слово, и о том, как это влияет на обучение эмбеддингов.

### Задание 2: Длина последовательности и RNN (8 минут)

Вернитесь к плейсхолдеру с `MAX_LEN` и попробуйте значения `128` и `512`. Сравните время обучения и точность.

- Как время обучения зависит от `MAX_LEN`?
- Всегда ли более длинная последовательность даёт лучший результат?
- Что происходит с информацией из начала длинного текста к моменту, когда RNN доходит до конца? Как это связано с архитектурой рекуррентного слоя?

### Задание 3: Двунаправленный RNN и поиск гиперпараметров (10 минут)

В блоке с гиперпараметрами модели попробуйте три конфигурации (можно временно поставить `NUM_EPOCHS = 2` для быстрой итерации):

| Конфигурация | EMBED_DIM | HIDDEN_DIM | NUM_LAYERS | BIDIRECTIONAL |
|--------------|-----------|------------|------------|---------------|
| A            | 64        | 64         | 1          | False         |
| B            | 128       | 256        | 1          | True          |
| C            | 256       | 128        | 2          | False         |

- Как количество параметров связано с точностью?
- Чем отличается обработка последовательности в двунаправленном RNN? Почему это может улучшать качество на задачах классификации?
- Какой вариант оказался лучшим на валидации и почему, на ваш взгляд?